# Day 23 — Solutions: Data Pipelines & Generators
Streaming CSV cleaning; compose filter/map/batch generators.

In [ ]:
from __future__ import annotations
from typing import Iterable, Iterator, Callable, TypeVar
import csv
from pathlib import Path
from itertools import islice

T = TypeVar('T')

def stream_csv(path: Path) -> Iterator[dict[str, str]]:
    with path.open(encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            yield row

def clean_rows(rows: Iterable[dict[str, str]]) -> Iterator[dict[str, str]]:
    for r in rows:
        r = {k: (v.strip() if isinstance(v, str) else v) for k, v in r.items()}
        if 'code' in r and isinstance(r['code'], str):
            r['code'] = r['code'].upper()
        yield r

def only_if(rows: Iterable[T], pred: Callable[[T], bool]) -> Iterator[T]:
    for r in rows:
        if pred(r):
            yield r

def mapper(rows: Iterable[T], fn: Callable[[T], T]) -> Iterator[T]:
    for r in rows:
        yield fn(r)

def batched(rows: Iterable[T], size: int) -> Iterator[list[T]]:
    it = iter(rows)
    while True:
        chunk = list(islice(it, size))
        if not chunk:
            return
        yield chunk

# Example (commented):
# pipeline = batched(
#     mapper(
#         only_if(clean_rows(stream_csv(Path('data.csv'))), lambda r: r.get('qty','0').isdigit()),
#         lambda r: {**r, 'qty': int(r['qty'])}
#     ),
#     size=500
# )
# for chunk in pipeline:
#     pass  # bulk_insert(chunk)